In [2]:
import pandas as pd
import glob
from scipy import signal
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import nn,optim
from torch.nn import functional as F
from scipy import stats
from sklearn import preprocessing
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn import datasets, linear_model
from sklearn.model_selection import cross_val_score
from tqdm import tqdm
from sklearn.utils import shuffle
import math
from keras.utils import np_utils
from scipy.stats import norm,kurtosis,skew



In [3]:
###
# All variables releated data are located here to make an order
###
le=preprocessing.LabelEncoder()
LABELS=['grazing','running','standing','trotting','walking-natural','walking-rider']
PATH="C:\\Users\\emirc\\Desktop\\AAR\\data\\csv\\datas/*" #Refer the path having the csv files
OUTPUT_LABEL='ActivityEncoded'
ACTIVITIES_MERGE={'running-natural': 'running',
                 'running-rider': 'running',
                  'trotting-natural': 'trotting',    #Having nearly the same meaning
                  'trotting-rider': 'trotting',} 
REMOVE_COLUMNS=['Mx','My','Mz','Gx','Gy','Gz','G3D','M3D','A3D'] #Columns to delete such as Gx Gy Gz
TIME_PERIODS = 200
STEP_DISTANCE = 100
FILES = sorted(glob.glob(PATH))
N_FEATURES=3
SUBJECTS=['Zafir','Driekus','Patron','Galoway','Happy']

In [4]:
def create_dataframe(files):
    result=pd.DataFrame()
    matching=[f for f in files if any(s in f for s in SUBJECTS)]
    
    for file in tqdm(matching):
        csv=pd.read_csv(file)
        csv['filename']=file
        result=result.append(csv)
    
    result.drop(REMOVE_COLUMNS, axis=1, inplace=True)
    result=select_activites(result)
    
    result[OUTPUT_LABEL]=le.fit_transform(result['label'].values.ravel())
    return result

In [5]:
def select_activites(df):
    df['label']=df['label'].replace(to_replace=['trotting-natural'], value='trotting')
    df['label']=df['label'].replace(to_replace=['trotting-rider'], value='trotting')
    df['label']=df['label'].replace(to_replace=['running-natural'], value='running')
    df['label']=df['label'].replace(to_replace=['running-rider'], value='running')
    result=df[df['label'].isin(LABELS)]
    return result

In [6]:
def filter(df):
    sos = signal.butter(N=3, Wn=30, btype='lowpass', fs=100, output='sos')
    df['Ax'] = signal.sosfilt(sos, df['Ax'])
    df['Ay'] = signal.sosfilt(sos, df['Ay'])
    df['Az'] = signal.sosfilt(sos, df['Az'])
    return df
    

In [7]:
def split_by_subject(df, name):
    test=df[df['filename'].str.contains(name)]
    train=df[~df['filename'].str.contains(name)]
    return train, test

In [8]:
def feature_scaling(data):
    train_x_max = data['Ax'].max()
    train_y_max = data['Ay'].max()
    train_z_max = data['Az'].max()

    pd.options.mode.chained_assignment = None

    #divide all 3 axis with the max value in the training set
    data['Ax'] = data['Ax'] / train_x_max
    data['Ay'] = data['Ay'] / train_y_max
    data['Az'] = data['Az'] / train_z_max

    return data

In [9]:
def create_windows(df, time_steps, step, label_name):
    windows = []
    labels = []
    for i in range(0, len(df)-time_steps, step):
        axs = df['Ax'].values[i: i + time_steps]
        ays = df['Ay'].values[i: i + time_steps]
        azs = df['Az'].values[i: i + time_steps]
        # Retrieve the most often used label in this segment
        label = stats.mode(df[label_name][i: i + time_steps])[0][0]
        windows.append([axs, ays, azs])
        labels.append(label)
    # Bring the segments into a better shape
    reshaped_windows = np.asarray(windows, dtype= np.float32).reshape(-1, time_steps, N_FEATURES)
    labels = np.asarray(labels)

    return reshaped_windows, labels

In [37]:
def getStatisticalFeatures(data,name):
    columns=['mean'+name,'std'+name,'median'+name,'cov'+name,'twf_perc'+name,'svf_perc'+name,'maxi'+name,'mini'+name,'skewness'+name,'kurt'+name]
    res=pd.DataFrame(columns=columns)
    for i, window in tqdm(enumerate(data)):
        maxi=np.amax(window)
        mini=np.amin(window)
        mean=np.mean(window, axis=None)
        std=np.std(window, axis=None)
        median=np.median(window, axis=None)
        cov=math.sqrt(std)
        twf_perc=np.percentile(window, 25, axis=None)
        svf_perc=np.percentile(window, 75, axis=None)
        skewness=skew(window)
        kurt=kurtosis(window)
        res=res.append({'mean'+name:mean,
                    'std'+name:std,
                    'median'+name:median,
                    'cov'+name:cov,
                    'twf_perc'+name:twf_perc,
                    'svf_perc'+name:svf_perc,
                    'maxi'+name:maxi,
                    'mini'+name:mini,
                    'skewness'+name:skewness,
                    'kurt'+name:kurt},ignore_index=True)
        
    return res


In [13]:
def reshape_input(x,shape):
    result=x.reshape(x.shape[0],shape)
    return result

In [14]:
def get_latent_representations(data, w=True):
    nr_of_windows=data.shape[0]
    data=data.reshape(nr_of_windows,0)
    tensor=torch.tensor(data)
    trained=[]
    for i, window in tqdm(enumerate(tensor)):
        reps=MODEL(torch.unsqueeze(window, 0))
        reps=torch.detach(reps[1])
        trained.append(reps.numpy())
        trained[i]=np.full((200,3), reps)
    trained=np.asarray(trained)
    return trained
    

In [15]:
def encode_output(y, classes):
    result=np_utils.to_categorical(y,classes)
    return result

In [16]:
def preprocess_training(df, input_shape,num_classes):
    train=feature_scaling(df)
    x_train, y_train=create_windows(train, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_train, y_train=shuffle(np.array(x_train),np.array(y_train))
    
    
    x_train=reshape_input(x_train,input_shape)
    x_train=x_train.astype('float32')
    y_train=y_train.astype('float32')
    y_train=encode_output(y_train,num_classes)
    return x_train, y_train


In [17]:
def preprocess_test(df, input_shape, num_classes):
    test=feature_scaling(df)
    x_test,y_test=create_windows(test, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_test=get_latent_representations(x_test)
    x_test=x_test.astype('float32')
    y_test=y_test.astype('float32')
    y_test=encode_output(y_test,num_classes)
    return x_test, y_test

In [18]:
def preprocess(df,test_subject):
    input_shape=(TIME_PERIODS*N_FEATURES)
    num_classes=len(LABELS)
    df=filter(df)
    train, test=split_by_subject(df, test_subject)
    x_train, y_train=preprocess_training(train, input_shape, num_classes)
    #x_test, y_test=preprocess_test(test, input_shape,num_classes)
    return x_train, y_train,input_shape,num_classes


In [19]:
def split_by_activity(df, activity):
    try:
        return df.loc[df['label']==activity]
    except KeyError:
        return

In [59]:
def featureExtraction(df, activity):
    df=split_by_activity(df,activity)
    train=feature_scaling(df)
    x_train, y_train=create_windows(train, TIME_PERIODS,STEP_DISTANCE,OUTPUT_LABEL)
    x_train, y_train=shuffle(np.array(x_train),np.array(y_train))
    #x_train=get_latent_representations(x_train)
    statistics=[]
    statistics.append(getStatisticalFeatures(x_train[::1],'_Ax'))
    statistics.append(getStatisticalFeatures(x_train[::2],'_AY'))
    statistics.append(getStatisticalFeatures(x_train[::3],'_Az'))
    return statistics

In [55]:
df=create_dataframe(FILES)
print("DATAFRAME")
print(df)

  0%|                                                                                           | 0/19 [00:00<?, ?it/s]C:\Users\emirc\AppData\Local\Temp/ipykernel_13396/1274980229.py:1: DtypeWarning: Columns (12) have mixed types.Specify dtype option on import or set low_memory=False.
  df=create_dataframe(FILES)
100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:38<00:00,  2.01s/it]


DATAFRAME
              Ax        Ay        Az            label  segment  subject  \
259705  4.649706 -0.660823  8.380005  walking-natural   147057        2   
259706  6.340072 -0.560263  8.145365  walking-natural   147057        2   
259707  6.804564  0.177177  6.991318  walking-natural   147057        2   
259708  7.609044  1.996835  5.257854  walking-natural   147057        2   
259709  7.982553  3.979305  3.826071  walking-natural   147057        2   
...          ...       ...       ...              ...      ...      ...   
326811 -0.201120 -4.391123 -6.148529  walking-natural   429615        3   
326812  0.062251 -4.683226 -5.741500  walking-natural   429615        3   
326813  0.914618 -6.186838 -5.152506  walking-natural   429615        3   
326814  0.881098 -7.020050 -4.817306  walking-natural   429615        3   
326815  0.900252 -7.508484 -4.395911  walking-natural   429615        3   

                                                 filename  ActivityEncoded  
259705  C:\U

In [60]:
#Index 0 --> Ax features, Index 1 -- > Ay features, Index 2 --> Az features
f_extracted=featureExtraction(df, 'walking-rider') 

15762it [00:46, 335.93it/s]
7881it [00:21, 359.92it/s]
5254it [00:14, 366.22it/s]


In [57]:
mergedDataFrame=f_extracted[0].merge(f_extracted[1],right_index=True,left_index=True)
mergedDataFrame=mergedDataFrame.merge(f_extracted[2],right_index=True,left_index=True)
print(mergedDataFrame.columns) ##To check whether the merge is successful or not 
print(mergedDataFrame)


Index(['mean_Ax', 'std_Ax', 'median_Ax', 'cov_Ax', 'twf_perc_Ax',
       'svf_perc_Ax', 'maxi_Ax', 'mini_Ax', 'skewness_Ax', 'kurt_Ax',
       'mean_AY', 'std_AY', 'median_AY', 'cov_AY', 'twf_perc_AY',
       'svf_perc_AY', 'maxi_AY', 'mini_AY', 'skewness_AY', 'kurt_AY',
       'mean_Az', 'std_Az', 'median_Az', 'cov_Az', 'twf_perc_Az',
       'svf_perc_Az', 'maxi_Az', 'mini_Az', 'skewness_Az', 'kurt_Az'],
      dtype='object')
       mean_Ax    std_Ax  median_Ax    cov_Ax  twf_perc_Ax  svf_perc_Ax  \
0     0.044619  0.078079   0.051042  0.279426    -0.011490     0.097380   
1     0.028548  0.072758   0.042771  0.269737    -0.041658     0.084666   
2    -0.047688  0.097882  -0.055762  0.312860    -0.100455     0.006991   
3     0.047378  0.068766   0.050576  0.262233    -0.011734     0.095378   
4    -0.042764  0.089902  -0.055778  0.299837    -0.097486     0.007477   
...        ...       ...        ...       ...          ...          ...   
5249  0.059636  0.053435   0.051565  0.23116